# Accident Detection from Dashcam Video — Preprocessing

<div dir="rtl">

این نوت‌بوک ورودی مدل تشخیص تصادف را می‌سازد. از هر ویدئو، از ابتدا تا انتها، هر **۰٫۲ ثانیه** یک فریم برداشته می‌شود و هر فریم به یک بردار ویژگی تبدیل می‌شود. خروجی برای هر ویدئو یک دنباله‌ی زمانی از این بردارهاست.

انتخاب فریم‌ها برای همه‌ی ویدئوها یکسان است و به زمان حادثه بستگی ندارد. از `time_of_event` فقط برای ساختن برچسب استفاده می‌شود (وجود یا نبود مقدار)؛ خود زمان‌ها نه در انتخاب فریم نقش دارند و نه وارد مدل می‌شوند.

اجرا: Colab با **T4 GPU**. اگر اتصال قطع شد، سلول‌ها را دوباره به ترتیب اجرا کنید؛ ویدئوهایی که ویژگی‌شان ذخیره شده دوباره پردازش نمی‌شوند.

</div>

## 1. Setup

In [5]:
import shutil
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu")

CLASS_COLORS = {"Accident": "#d62728", "No accident": "#1f77b4"}

Device: Tesla T4


<div dir="rtl">

ویدئوها فقط موقتاً روی دیسک Colab دانلود و بعد از پردازش پاک می‌شوند؛ ویژگی‌ها و تقسیم داده روی Drive می‌مانند.

</div>

In [6]:
from google.colab import drive
drive.mount("/content/drive")

REPO_ID = "nexar-ai/nexar_collision_prediction"
SEED = 42
STEP = 0.2                                   # seconds between sampled frames
CLIP_NAME = "openai/clip-vit-base-patch16"
DINO_NAME = "facebook/dinov2-base"
CLIP_HW, DINO_HW = (224, 400), (224, 392)    # 16:9 inputs, no square crop
FLOW_HW = (108, 192)                         # small grayscale frames for optical flow
BATCH = 64

PROJECT_DIR = Path("/content/drive/MyDrive/accident_detection")
FEAT_DIR = PROJECT_DIR / "features"
VIDEO_DIR = Path("/content/videos")
for folder in (PROJECT_DIR, FEAT_DIR, VIDEO_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print(f"Drive free space: {shutil.disk_usage(PROJECT_DIR).free / 1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive free space: 4.3 GB


<div dir="rtl">

دسترسی به دیتاست روی Hugging Face با token است. token از Colab Secrets با نام `HF_TOKEN` خوانده می‌شود و اگر نبود، پرسیده می‌شود.

</div>

In [7]:
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face token: ").strip()
api = HfApi(token=HF_TOKEN)

## 2. Dataset Inventory

<div dir="rtl">

فهرست فایل‌ها و حجمشان از API گرفته می‌شود؛ هنوز ویدئویی دانلود نمی‌شود.

</div>

In [8]:
from huggingface_hub.hf_api import RepoFile

tree = api.list_repo_tree(REPO_ID, repo_type="dataset", recursive=True)
inventory = pd.DataFrame([{"path": f.path, "size_mb": f.size / 1e6} for f in tree if isinstance(f, RepoFile)])

videos = inventory[inventory["path"].str.endswith(".mp4")].copy()
videos[["split", "cls", "file_name"]] = videos["path"].str.split("/", expand=True)
display(videos.groupby(["split", "cls"])["size_mb"].agg(videos="size", mean_mb="mean", total_mb="sum").round(1))

videos  mean_mb  total_mb
split        cls                                
test-private negative     339      4.2    1432.0
             positive     338      4.5    1536.1
test-public  negative     333      4.3    1441.3
             positive     334      4.3    1440.0
train        negative     750     17.1   12840.6
             positive     750     16.9   12689.0

## 3. Metadata and Labels

<div dir="rtl">

برچسب از وجود مقدار در `time_of_event` ساخته می‌شود (۱ = تصادف یا شبه‌تصادف) و با نام پوشه کنترل می‌شود. ستون‌های زمانی فقط در `metadata_all.csv` می‌مانند و به فایل تقسیم داده نمی‌روند.

</div>

In [9]:
parts = []
for path in sorted(inventory.loc[inventory["path"].str.endswith("metadata.csv"), "path"]):
    local = hf_hub_download(REPO_ID, path, repo_type="dataset", token=HF_TOKEN, local_dir=PROJECT_DIR / "metadata")
    split, cls, _ = path.split("/")
    parts.append(pd.read_csv(local, dtype={"file_name": str}).assign(split=split, cls=cls))

meta = pd.concat(parts, ignore_index=True)
meta["label"] = meta["time_of_event"].notna().astype(int)
assert (meta["label"] == meta["cls"].eq("positive")).all()   # label agrees with the folder name

meta["video_id"] = meta["file_name"].str.replace(".mp4", "", regex=False)
meta["path"] = meta["split"] + "/" + meta["cls"] + "/" + meta["file_name"]
meta = meta.merge(videos[["path", "size_mb"]], on="path")
meta[["light_conditions", "weather", "scene"]] = meta[["light_conditions", "weather", "scene"]].fillna("Unknown")
meta["class"] = meta["label"].map({0: "No accident", 1: "Accident"})
meta.to_csv(PROJECT_DIR / "metadata_all.csv", index=False)

display(meta.groupby(["split", "class"]).size().unstack())

metadata.csv:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

metadata.csv:   0%|          | 0.00/17.0k [00:00<?, ?B/s]

metadata.csv:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

metadata.csv:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

metadata.csv:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

metadata.csv:   0%|          | 0.00/33.3k [00:00<?, ?B/s]

class,Accident,No accident
split,,
test-private,338,339
test-public,334,333
train,750,750


## 4. Fixed Split

<div dir="rtl">

از ۱۵۰۰ ویدئوی `train`، ۳۰۰ ویدئو به‌عنوان **test** کنار گذاشته می‌شوند و فقط یک بار، در ارزیابی نهایی، استفاده می‌شوند. ۱۲۰۰ ویدئوی باقی‌مانده (**dev**) به ۵ fold تقسیم می‌شوند و همه‌ی تحلیل‌ها و مقایسه‌ها فقط روی dev انجام می‌شوند. تقسیم بر اساس برچسب و نوع صحنه stratify می‌شود و فقط یک بار ساخته و ذخیره می‌شود.

</div>

In [10]:
from sklearn.model_selection import train_test_split, StratifiedKFold

train = meta[meta["split"] == "train"].reset_index(drop=True)
split_path = PROJECT_DIR / "splits.csv"

if split_path.exists():
    splits = pd.read_csv(split_path, dtype={"video_id": str})
    print("Loaded the saved split.")
else:
    # rare scenes are merged so every stratum has enough videos
    scene_counts = train["scene"].value_counts()
    scene_group = train["scene"].where(train["scene"].map(scene_counts) >= 30, "Rare")
    strata = train["label"].astype(str) + "_" + scene_group

    dev_idx, _ = train_test_split(np.arange(len(train)), test_size=300, stratify=strata, random_state=SEED)
    fold = np.full(len(train), -1)   # -1 = test
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for k, (_, val_pos) in enumerate(skf.split(dev_idx, strata.iloc[dev_idx])):
        fold[dev_idx[val_pos]] = k

    keep = ["video_id", "split", "path", "label", "light_conditions", "weather", "scene", "size_mb"]
    splits = train[keep].assign(fold=fold)
    splits.to_csv(split_path, index=False)

dev = splits[splits["fold"] >= 0]
display(pd.crosstab(splits["fold"], splits["label"], margins=True))

Loaded the saved split.


label,0,1,All
fold,,,
-1,151,149,300
0,121,119,240
1,120,120,240
2,120,120,240
3,119,121,240
4,119,121,240
All,750,750,1500


## 5. Data Overview (dev)

In [11]:
dev_meta = train[train["video_id"].isin(dev["video_id"])]

counts = (dev_meta.melt(id_vars="class", value_vars=["light_conditions", "weather", "scene"],
                        var_name="field", value_name="value")
          .groupby(["field", "value", "class"]).size().reset_index(name="videos"))
fig = px.bar(counts, x="value", y="videos", color="class", barmode="group", facet_col="field",
             color_discrete_map=CLASS_COLORS, title="Environment by class (dev)")
fig.update_xaxes(matches=None, title=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

pos = dev_meta[dev_meta["label"] == 1]
fig = px.histogram(pos, x="time_of_event", nbins=40, color_discrete_sequence=["#d62728"],
                   title="Time of the event inside positive dev videos")
fig.update_layout(xaxis_title="seconds from video start", yaxis_title="videos")
fig.show()
print(f"Events between 18 s and 22 s: {pos['time_of_event'].between(18, 22).mean():.0%}")

Events between 18 s and 22 s: 87%


<div dir="rtl">

زمان حادثه تقریباً همیشه حوالی ثانیه‌ی ۲۰ است. اگر فریم‌ها بر اساس این زمان انتخاب شوند، مدل از راه نمونه‌برداری به جواب می‌رسد، در حالی که در استفاده‌ی واقعی زمان حادثه معلوم نیست. برای همین نمونه‌برداری یکنواخت و روی کل ویدئوست، و مدل مرحله‌ی بعد هم نباید به جایگاه مطلق زمان (مثلاً «ثانیه‌ی ۲۰») حساس باشد.

</div>

## 6. Shortcut Check

<div dir="rtl">

آیا متادیتا یا حجم فایل به‌تنهایی کلاس را لو می‌دهد؟ یک logistic regression روی foldهای dev. مدل ویدئویی باید به‌وضوح از این عددها بالاتر برود.

</div>

In [12]:
from sklearn.model_selection import PredefinedSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

dev_cv = PredefinedSplit(dev["fold"])
baselines = {
    "metadata": (make_pipeline(OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=1000)),
                 dev[["light_conditions", "weather", "scene"]]),
    "file size": (make_pipeline(StandardScaler(), LogisticRegression()), dev[["size_mb"]]),
}
for name, (model, X) in baselines.items():
    s = cross_validate(model, X, dev["label"], cv=dev_cv, scoring=["accuracy", "roc_auc"])
    print(f"{name}: accuracy {s['test_accuracy'].mean():.3f} | ROC-AUC {s['test_roc_auc'].mean():.3f}")

metadata: accuracy 0.619 | ROC-AUC 0.642
file size: accuracy 0.514 | ROC-AUC 0.502


## 7. Frame Sampling and Features

<div dir="rtl">

- **نمونه‌برداری:** هر ۰٫۲ ثانیه یک فریم، یعنی حدود ۲۰۰ فریم برای ۴۰ ثانیه. لحظه‌ی برخورد کسری از ثانیه است و با فاصله‌ی بیشتر ممکن است بین دو نمونه بیفتد.
- **ویژگی تصویری:** دو encoder از پیش آموزش‌دیده و frozen: **CLIP ViT-B/16** (معنای صحنه) و **DINOv2 ViT-B/14** (جزئیات مکانی). با ۱۵۰۰ ویدئو و یک برچسب برای هر کدام، یادگرفتن ویژگی تصویری از صفر عملی نیست؛ فقط مدل زمانیِ مرحله‌ی بعد آموزش می‌بیند.
- **نسبت ۱۶:۹:** فریم‌ها مربعی نمی‌شوند؛ برش مربعی حدود ۴۴٪ عرض تصویر را حذف می‌کند و برخوردهای جانبی همان‌جاست.
- **حرکت:** در شبه‌تصادف خسارتی دیده نمی‌شود و نشانه‌ی اصلی ترمز یا انحراف ناگهانی است. optical flow بین هر فریم نمونه و فریم بعدی‌اش (۱/۳۰ ثانیه) حساب و به ۶ عدد خلاصه می‌شود:

| ویژگی | معنا |
|---|---|
| `flow_mean` | شدت کلی حرکت |
| `flow_bottom` | حرکت یک‌سوم پایین تصویر (جاده)؛ تقریبی از سرعت |
| `expansion` | باز شدن تصویر از مرکز؛ حرکت رو به جلو یا نزدیک شدن جسم |
| `flow_x` | انحراف به چپ و راست |
| `flow_y` | تکان عمودی؛ ضربه یا ترمز شدید |
| `pixel_change` | تغییر روشنایی بین دو فریم |

</div>

In [13]:
import transformers
from transformers import AutoImageProcessor, AutoModel, CLIPVisionModelWithProjection

transformers.logging.set_verbosity_error()
clip_model = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME).to(device).eval()
dino_model = AutoModel.from_pretrained(DINO_NAME).to(device).eval()


def norm_stats(name):
    processor = AutoImageProcessor.from_pretrained(name)
    mean = torch.tensor(processor.image_mean, device=device).view(1, 3, 1, 1)
    std = torch.tensor(processor.image_std, device=device).view(1, 3, 1, 1)
    return mean, std


CLIP_NORM, DINO_NORM = norm_stats(CLIP_NAME), norm_stats(DINO_NAME)

config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

In [14]:
FH, FW = FLOW_HW
yy, xx = np.mgrid[0:FH, 0:FW].astype(np.float32)
dx, dy = xx - FW / 2, yy - FH / 2
r = np.hypot(dx, dy) + 1e-6
RX, RY = dx / r, dy / r          # unit vectors pointing away from the image center
BOTTOM = yy > FH * 2 / 3         # bottom third of the image (road)
MOTION_NAMES = ["flow_mean", "flow_bottom", "expansion", "flow_x", "flow_y", "pixel_change"]


def motion_features(gray0, gray1, dis, fps):
    flow = dis.calc(gray0, gray1, None) * fps    # pixels per second
    u, v = flow[..., 0], flow[..., 1]
    mag = np.hypot(u, v)
    change = np.abs(gray1.astype(np.float32) - gray0).mean() / 255
    return np.array([mag.mean(), mag[BOTTOM].mean(), (u * RX + v * RY).mean(),
                     u.mean(), v.mean(), change], dtype=np.float32)


def read_video(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not (fps > 0 and n_frames > 1):
        raise ValueError(f"Cannot read {path}")

    # one frame every STEP seconds over the whole video; each one also needs its next frame for the flow
    idx = np.rint(np.arange(0, n_frames / fps, STEP) * fps).astype(int)
    wanted = set(idx[idx < n_frames - 1].tolist())
    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_FAST)

    clip_frames, dino_frames, motion, times = [], [], [], []
    prev_gray, pos = None, 0
    while cap.grab():
        if pos in wanted or pos - 1 in wanted:
            frame = cap.retrieve()[1]
            gray = cv2.cvtColor(cv2.resize(frame, (FW, FH), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
            if pos - 1 in wanted:                       # the frame right after a sample
                motion.append(motion_features(prev_gray, gray, dis, fps))
            if pos in wanted:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                clip_frames.append(cv2.resize(rgb, CLIP_HW[::-1], interpolation=cv2.INTER_AREA))
                dino_frames.append(cv2.resize(rgb, DINO_HW[::-1], interpolation=cv2.INTER_AREA))
                times.append(pos / fps)
                prev_gray = gray
        pos += 1
    cap.release()

    n = len(motion)    # the last sample may have no next frame
    return {"clip_frames": np.stack(clip_frames[:n]), "dino_frames": np.stack(dino_frames[:n]),
            "motion": np.stack(motion), "times": np.array(times[:n], dtype=np.float32),
            "fps": fps, "n_frames": n_frames}


@torch.no_grad()
def embed(frames, model, norm, kind):
    out = []
    for i in range(0, len(frames), BATCH):
        x = torch.from_numpy(frames[i:i + BATCH]).to(device)
        x = (x.permute(0, 3, 1, 2).float() / 255 - norm[0]) / norm[1]
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            if kind == "clip":
                feats = model(pixel_values=x, interpolate_pos_encoding=True).image_embeds
            else:
                tokens = model(pixel_values=x).last_hidden_state
                feats = torch.cat([tokens[:, 0], tokens[:, 1:].mean(1)], dim=1)   # CLS + mean of patches
        out.append(feats.float().cpu())
    return torch.cat(out).numpy().astype(np.float16)


def encode_video(data):
    return {"clip": embed(data["clip_frames"], clip_model, CLIP_NORM, "clip"),
            "dino": embed(data["dino_frames"], dino_model, DINO_NORM, "dino"),
            "motion": data["motion"], "times": data["times"],
            "fps": data["fps"], "n_frames": data["n_frames"]}

In [15]:
def feature_path(split, video_id):
    return FEAT_DIR / split / f"{video_id}.npz"


def save_features(feats, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_name(target.stem + ".tmp.npz")
    np.savez(tmp, **feats)
    tmp.rename(target)    # a half-written file never gets the final name


def prepare(path):
    """Download one video, read its frames, delete the video. Returns None on failure."""
    try:
        video = Path(hf_hub_download(REPO_ID, path, repo_type="dataset", token=HF_TOKEN, local_dir=VIDEO_DIR))
    except Exception as error:
        print(f"Download failed: {path} ({error})")
        return None
    try:
        return read_video(video)
    except Exception as error:
        print(f"Read failed: {path} ({error})")
        return None
    finally:
        video.unlink(missing_ok=True)

## 8. Test on Two Videos

<div dir="rtl">

کل مسیر روی یک ویدئوی تصادف و یک ویدئوی عادی امتحان می‌شود. در نمودار حرکت، خط‌چین زمان حادثه فقط برای مقایسه‌ی چشمی است و در هیچ محاسبه‌ای نیست.

</div>

In [16]:
demo = dev.groupby("label").head(1)
results = {}
for r in demo.itertuples(index=False):
    started = time.time()
    data = prepare(r.path)
    feats = encode_video(data)
    results[r.label] = (r, data)
    print(f"{r.path}: {len(data['times'])} frames | {time.time() - started:.1f} s |",
          {k: v.shape for k, v in feats.items() if np.ndim(v)})

train/negative/01924.mp4: reconstructing file:   0%|          |  0.00B / 23.6MB            

train/negative/01924.mp4: downloading bytes:           |  0.00B            

train/negative/01924.mp4: 200 frames | 15.2 s | {'clip': (200, 512), 'dino': (200, 1536), 'motion': (200, 6), 'times': (200,)}


train/positive/00822.mp4: reconstructing file:   0%|          |  0.00B / 17.4MB            

train/positive/00822.mp4: downloading bytes:           |  0.00B            

train/positive/00822.mp4: 203 frames | 12.9 s | {'clip': (203, 512), 'dino': (203, 1536), 'motion': (203, 6), 'times': (203,)}


In [17]:
r, data = results[1]
show = np.linspace(0, len(data["times"]) - 1, 8).round().astype(int)
fig = px.imshow(data["clip_frames"][show], facet_col=0, facet_col_wrap=4, binary_string=True,
                title=f"Sampled frames of an accident video ({r.video_id})")
fig.for_each_annotation(lambda a: a.update(text=f"{data['times'][show][int(a.text.split('=')[1])]:.1f} s"))
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(height=430)
fig.show()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=["Accident video", "Normal driving video"])
for row, label in [(1, 1), (2, 0)]:
    d = results[label][1]
    for i, name in enumerate(["flow_bottom", "expansion", "flow_y"]):
        fig.add_trace(go.Scatter(x=d["times"], y=d["motion"][:, MOTION_NAMES.index(name)], name=name,
                                 line=dict(color=px.colors.qualitative.Plotly[i]), showlegend=row == 1),
                      row=row, col=1)
event_time = meta.set_index("path").loc[r.path, "time_of_event"]   # for the eye only
fig.add_vline(x=event_time, line_dash="dash", line_color="#d62728", row=1, col=1)
fig.update_layout(height=550, title="Motion features over time")
fig.update_xaxes(title_text="seconds", row=2, col=1)
fig.update_yaxes(title_text="pixels / s")
fig.show()

## 9. Feature Extraction

<div dir="rtl">

دو thread دانلود و خواندن ویدئوها را روی CPU انجام می‌دهند و هم‌زمان GPU ویدئوهای آماده را encode می‌کند. ویژگی‌های هر ویدئو جدا روی Drive ذخیره و خود ویدئو پاک می‌شود (حدود ۰٫۸ مگابایت به‌جای حدود ۱۷ مگابایت)؛ به همین دلیل می‌شود از هر ۱۵۰۰ ویدئو استفاده کرد. ویدئوهایی که قبلاً ذخیره شده‌اند رد می‌شوند و ویدئوهای ناموفق با اجرای دوباره‌ی سلول دوباره امتحان می‌شوند.

</div>

In [18]:
def extract_all(table, chunk=8):
    todo = [r for r in table.itertuples(index=False) if not feature_path(r.split, r.video_id).exists()]
    print(f"Already done: {len(table) - len(todo)} | to extract: {len(todo)}")
    failed = []
    with ThreadPoolExecutor(max_workers=2) as pool:
        for start in tqdm(range(0, len(todo), chunk), desc=f"chunks of {chunk} videos"):
            rows = todo[start:start + chunk]
            for r, data in zip(rows, pool.map(prepare, [r.path for r in rows])):
                if data is None:
                    failed.append(r.path)
                else:
                    save_features(encode_video(data), feature_path(r.split, r.video_id))
    return failed


failed = extract_all(splits)
print("Failed:", failed)    # rerun this cell to retry them

Already done: 1500 | to extract: 0


chunks of 8 videos: 0it [00:00, ?it/s]

Failed: []


## 10. Verification

<div dir="rtl">

همه‌ی فایل‌ها خوانده و کنترل می‌شوند: ابعاد، نبود مقدار نامعتبر، صعودی بودن زمان‌ها و پوشش کامل ویدئو. بعد طول ویدئو و fps در دو کلاس مقایسه می‌شوند تا مطمئن شویم کلاس را لو نمی‌دهند.

</div>

In [19]:
def pool_embedding(x):
    x = x.astype(np.float32)
    return np.concatenate([x.mean(0), x.max(0), x.std(0)])


def pool_motion(x):
    return np.concatenate([x.mean(0), x.std(0), x.min(0), x.max(0), np.abs(np.diff(x, axis=0)).max(0)])


info, problems, pooled = [], [], {"CLIP": [], "DINOv2": [], "Motion": []}
for r in tqdm(splits.itertuples(index=False), total=len(splits)):
    try:
        with np.load(feature_path(r.split, r.video_id)) as f:
            T = len(f["times"])
            assert f["clip"].shape == (T, 512) and f["dino"].shape == (T, 1536) and f["motion"].shape == (T, 6)
            assert all(np.isfinite(f[k]).all() for k in ("clip", "dino", "motion"))
            assert (np.diff(f["times"]) > 0).all()
            duration = float(f["n_frames"] / f["fps"])
            info.append({"video_id": r.video_id, "frames": T, "fps": float(f["fps"]), "duration_s": duration,
                         "coverage": min(1, (f["times"][-1] + STEP) / duration)})
            pooled["CLIP"].append(pool_embedding(f["clip"]))
            pooled["DINOv2"].append(pool_embedding(f["dino"]))
            pooled["Motion"].append(pool_motion(f["motion"]))
    except Exception as error:
        problems.append((r.path, repr(error)))

print(f"Verified: {len(info)} | problems: {len(problems)}")
assert not problems, problems[:5]    # delete these files, then rerun the extraction cell

info = splits[["video_id", "label", "fold"]].merge(pd.DataFrame(info), on="video_id")
info["class"] = info["label"].map({0: "No accident", 1: "Accident"})
info.to_csv(PROJECT_DIR / "video_info.csv", index=False)
print("Videos with less than 95% coverage:", (info["coverage"] < 0.95).sum())

  0%|          | 0/1500 [00:00<?, ?it/s]

Verified: 1500 | problems: 0
Videos with less than 95% coverage: 0


In [20]:
dev_info = info[info["fold"] >= 0]
display(dev_info.groupby("class")[["fps", "duration_s", "frames"]].agg(["mean", "min", "max"]).round(2))

fig = px.histogram(dev_info, x="duration_s", color="class", barmode="overlay", nbins=60, opacity=0.7,
                   color_discrete_map=CLASS_COLORS, title="Video duration by class (dev)")
fig.update_layout(xaxis_title="seconds")
fig.show()

s = cross_validate(make_pipeline(StandardScaler(), LogisticRegression()), dev_info[["duration_s"]],
                   dev_info["label"], cv=PredefinedSplit(dev_info["fold"]), scoring=["accuracy", "roc_auc"])
print(f"Duration only: accuracy {s['test_accuracy'].mean():.3f} | ROC-AUC {s['test_roc_auc'].mean():.3f}")

fps             duration_s              frames         
              mean   min   max       mean   min   max    mean min  max
class                                                                 
Accident     30.08  25.3  31.0      38.31  18.0  60.0  191.79  90  300
No accident  30.00  23.6  30.6      37.20  15.0  60.0  186.19  75  300

Duration only: accuracy 0.527 | ROC-AUC 0.530


## 11. Quick Feature Check (Linear Probe)

<div dir="rtl">

این مدل نهایی نیست. ویژگی‌ها در طول زمان با mean، max و std خلاصه می‌شوند و یک logistic regression روی foldهای dev آموزش می‌بیند. ترتیب زمانی نادیده گرفته می‌شود و لحظه‌ی کوتاه حادثه در میانگین ۴۰ ثانیه گم می‌شود؛ پس این عدد فقط نشان می‌دهد ویژگی‌ها چقدر اطلاعات دارند و مدل زمانی باید از آن بالاتر برود.

</div>

In [21]:
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X_sets = {name: np.stack(v) for name, v in pooled.items()}
X_sets["All"] = np.hstack(list(X_sets.values()))
is_dev = (splits["fold"] >= 0).to_numpy()
y_dev = splits.loc[is_dev, "label"].to_numpy()
dev_cv = PredefinedSplit(splits.loc[is_dev, "fold"])

rows = []
for name, X in X_sets.items():
    model = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
                         {"logisticregression__C": [1e-3, 1e-2, 1e-1, 1]}, cv=3, scoring="roc_auc")
    prob = cross_val_predict(model, X[is_dev], y_dev, cv=dev_cv, method="predict_proba")[:, 1]
    rows.append({"features": name, "accuracy": accuracy_score(y_dev, prob >= 0.5),
                 "f1": f1_score(y_dev, prob >= 0.5), "roc_auc": roc_auc_score(y_dev, prob)})

probe = pd.DataFrame(rows).round(3)
probe.to_csv(PROJECT_DIR / "linear_probe.csv", index=False)
display(probe)
px.bar(probe, x="features", y="accuracy", text="accuracy", range_y=[0.5, 1],
       title="Linear probe on time-pooled features (dev, 5 folds)").show()

,features,accuracy,f1,roc_auc
0,CLIP,0.692,0.690,0.755
1,DINOv2,0.696,0.691,0.771
2,Motion,0.682,0.692,0.730
3,All,0.699,0.697,0.779


## 12. Official Test Clips (optional)

<div dir="rtl">

کلیپ‌های `test-public` و `test-private` حدود ۱۰ ثانیه‌اند و ۰٫۵ تا ۱٫۵ ثانیه **قبل** از حادثه تمام می‌شوند؛ یعنی سؤالشان پیش‌بینی است، نه تشخیص. ویژگی‌هایشان با همان قاعده برای مسئله‌ی جدید (پیش‌بینی برخورد) استخراج می‌شود و در هیچ تصمیمی برای مدل تشخیص استفاده نمی‌شوند.

</div>

In [22]:
official = meta[meta["split"] != "train"]
print("Failed:", extract_all(official))

Already done: 1344 | to extract: 0


chunks of 8 videos: 0it [00:00, ?it/s]

Failed: []


## Outputs

<div dir="rtl">

در پوشه‌ی `accident_detection` روی Drive:
- `metadata_all.csv`: متادیتای کامل (ستون‌های زمانی فقط برای تحلیل)
- `splits.csv`: foldهای dev (۰ تا ۴) و test (‎-1)
- `features/<split>/<video_id>.npz`: `clip` (T×512)، `dino` (T×1536)، `motion` (T×6)، `times`، `fps` و `n_frames`
- `video_info.csv` و `linear_probe.csv`

</div>